In [ ]:
# CNN2 features to be concatenated

from nupack import *
import RNA
import math
import itertools

def DNA_to_RNA(DNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'T': 'U'}
    return ''.join(match.get(base, base) for base in (DNA))
            
def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def RNA_reverse_complement(RNA):
    complement = {'A': 'U', 'C': 'G', 'G': 'C', 'U': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(RNA))

def get_segments_interrupted_by_plus(input_string: str) -> list:
    # Split the string by the "+" character
    segments = input_string.split("+")
    
    # Remove empty segments if any (e.g., if the string starts or ends with "+")
    segments = [segment for segment in segments if segment]
    
    return segments

def find_substring_positions(substring, long_string):

    positions = []
    start = 0
    while True:
        index = long_string.find(substring, start)
        if index == -1:  # No more occurrences found.
            break
        positions.append(index)
        start = index + 1  # Move past the last found index.
    return positions

file_paths = [
    'Feature_CNN2_to_be_concatenated.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  

# Define NUPACK model-same as CRISPR trans-cleavage reaction
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('merged_Cas13a_1e11_full_guide_sequences_unique_before_19min.txt', 'r')    
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('merged_Cas13a_1e11_target_sequences_noPFS_unique_before_19min.txt', 'r')    
lines = file2.readlines()
full_target_array = [line.strip() for line in lines]
file3 = open('merged_Cas13a_1e11_target_sequences_noPFS_unique_before_19min.txt', 'r')    
lines = file3.readlines()
truncated_target_array = [line.strip() for line in lines]
file4 = open('merged_Cas13a_1e11_full_target_sequences_unique_before_19min.txt', 'r')   
lines = file4.readlines()
full_target_array_complete = [DNA_to_RNA(line.strip()) for line in lines]

for i in range (0, len(guide_array)):

    # Initialize struct_array
    struct_array = []
    struct_array_unit = []

    guide = guide_array[i]
    target_full = full_target_array[i]
    target_full_complete = full_target_array_complete[i]
    target_truncated = truncated_target_array[i]

    start_positions = find_substring_positions(target_truncated, target_full_complete)
    start_position = start_positions[0]

    # Compute suboptimal structures and energy
    subopt_structures_guide_bh = subopt(strands=guide, energy_gap=3, model=my_model_RNA)  
    subopt_structures_target_bh = subopt(strands=target_full_complete, energy_gap=0.1, model=my_model_RNA)
    subopt_structures_hybrid_ah = subopt(strands=[guide, target_truncated], energy_gap=3, model=my_model_RNA)

    # Guide structure before hybridization
    for j in range (0, 64):
        
        try:
            if str(subopt_structures_guide_bh[0].structure)[j] == '.':
                to_be_added = ([1, 0, 0, 0])
            elif str(subopt_structures_guide_bh[0].structure)[j] == '(':
                to_be_added = ([0, 1, 0, 0])
            elif str(subopt_structures_guide_bh[0].structure)[j] == ')':
                to_be_added = ([0, 0, 1, 0])
            else:
                to_be_added = ([0, 0, 0, 0])
                
        except IndexError:
            to_be_added = ([0, 0, 0, 0])

        struct_array_unit.append(to_be_added)
        
    # Target structure before hybridization 
    target_bh = get_segments_interrupted_by_plus(str(subopt_structures_target_bh[0].structure)[start_position:start_position+28])[0]
    
    for j in range (0, 28):
        
        try:
            if target_bh[j] == '.':
                to_be_added = ([1, 0, 0, 0])
            elif target_bh[j] == '(':
                to_be_added = ([0, 1, 0, 0])
            elif target_bh[j] == ')':
                to_be_added = ([0, 0, 1, 0])
            else:
                to_be_added = ([0, 0, 0, 0])
            
        except IndexError:
             to_be_added = ([0, 0, 0, 0])
        
        struct_array_unit.append(to_be_added)

         
    # Hybrid structure after hybridization 
    hybrid_ah_1 = get_segments_interrupted_by_plus(str(subopt_structures_hybrid_ah[0].structure))[0]
    hybrid_ah_2 = get_segments_interrupted_by_plus(str(subopt_structures_hybrid_ah[0].structure))[1]

    for j in range (0, 64):
        
        try:
            if hybrid_ah_1[j] == '.':
                to_be_added = ([1, 0, 0, 0])
            elif hybrid_ah_1[j] == '(':
                to_be_added = ([0, 1, 0, 0])
            elif hybrid_ah_1[j] == ')':
                to_be_added = ([0, 0, 1, 0])
            else:
                to_be_added = ([0, 0, 0, 0])
            
        except IndexError:
             to_be_added = ([0, 0, 0, 0])
        
        struct_array_unit.append(to_be_added)

     
    for j in range (0, 28):
        
        try:
            if hybrid_ah_2[j] == '.':
                to_be_added = ([1, 0, 0, 0])
            elif hybrid_ah_2[j] == '(':
                to_be_added = ([0, 1, 0, 0])
            elif hybrid_ah_2[j] == ')':
                to_be_added = ([0, 0, 1, 0])
            else:
                to_be_added = ([0, 0, 0, 0])
            
        except IndexError:
             to_be_added = ([0, 0, 0, 0])
        
        struct_array_unit.append(to_be_added)

            
    struct_array_to_append = [struct_array_unit]
    struct_array.append(struct_array_to_append)

    struct_array = list(itertools.chain.from_iterable(struct_array))
    
    # Open a file in write mode
    with open('Feature_CNN2_to_be_concatenated.txt', 'a') as file:
        # Iterate over each row in the 2D array
        for row in struct_array:
            # Convert each element to a string and join them with spaces
            file.write(' '.join(map(str, row)) + '\n')
        file.write('\n')

print('-------------')

In [ ]:
file1 = "Feature_CNN1.txt"
file2 = "Feature_CNN2_to_be_concatenated.txt"
output_file = "Feature_CNN1_CNN2_concatenated.txt"

with open(file1, "r") as f1, open(file2, "r") as f2:
    lines1 = f1.readlines()
    lines2 = f2.readlines()

if len(lines1) != len(lines2):
    raise ValueError(f"Different line numbers: {len(lines1)} vs {len(lines2)}")

with open(output_file, "w") as out:
    for line1, line2 in zip(lines1, lines2):
        line1 = line1.strip()
        line2 = line2.strip()

        out.write(line1 + " " + line2 + "\n")

print("Saved:", output_file)